In [2]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import re
import time

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}

def parse_attributes(attr_text):
    """'2014, 2.0 L, 112 021 km' mətnini İl, Mühərrik və Yürüşə ayırır."""
    year, engine, mileage = None, None, None
    if not attr_text:
        return year, engine, mileage
    
    parts = [p.strip() for p in attr_text.split(",")]
    
    for part in parts:
        if re.search(r'\b(19\d{2}|20\d{2})\b', part) and not year:
            year = int(re.search(r'\b(19\d{2}|20\d{2})\b', part).group(0))
        elif 'km' in part.lower():
            digits = re.sub(r'[^\d]', '', part)
            mileage = int(digits) if digits else None
        elif 'L' in part or 'l' in part:
            engine = part

    return year, engine, mileage

def scrape_turbo_5k(target_count=5000, output_file="turbo_az_5k_dataset.csv"):
    cars_data = []
    page = 1
    
    print(f"Hədəf: {target_count} elan. Çəkilməyə başlanılır...\n")

    while len(cars_data) < target_count:
        url = f"https://turbo.az/autos?page={page}"
        res = requests.get(url, headers=HEADERS)
        
        if res.status_code != 200:
            print(f"Səhifə {page} yüklənə bilmədi (Status Code: {res.status_code}). 3 saniyə gözlənilir...")
            time.sleep(3)
            continue

        soup = BeautifulSoup(res.text, "html.parser")
        items = soup.find_all("div", class_="products-i")

        if not items:
            print("Daha çox elan tapılmadı və ya kataloq bitdi.")
            break

        for item in items:
            if len(cars_data) >= target_count:
                break

            # Marka və Model
            title_elem = item.find("div", class_="products-i__name")
            title = title_elem.text.strip() if title_elem else ""
            make = title.split()[0] if title else None
            model = " ".join(title.split()[1:]) if title else None

            # Qiymət və Valyuta
            price_elem = item.find("div", class_="product-price") or item.find("div", class_="products-i__price")
            price_text = price_elem.text.strip() if price_elem else ""
            
            currency = "AZN"
            if "$" in price_text or "USD" in price_text:
                currency = "USD"
            elif "€" in price_text or "EUR" in price_text:
                currency = "EUR"
                
            price_digits = re.sub(r'[^\d]', '', price_text)
            price_val = float(price_digits) if price_digits else None

            # Atributlar (İl, Mühərrik, Yürüş)
            attr_elem = item.find("div", class_="products-i__attributes")
            attr_text = attr_elem.text.strip() if attr_elem else ""
            year, engine, mileage = parse_attributes(attr_text)

            # Şəhər
            datetime_elem = item.find("div", class_="products-i__datetime")
            city_datetime = datetime_elem.text.strip() if datetime_elem else ""
            city = city_datetime.split(",")[0].strip() if "," in city_datetime else city_datetime.split()[0] if city_datetime else None

            # URL
            link_elem = item.find("a", class_="products-i__link")
            link = "https://turbo.az" + link_elem["href"] if link_elem and "href" in link_elem.attrs else None

            cars_data.append({
                "make": make,
                "model": model,
                "price_value": price_val,
                "price_currency": currency,
                "year": year,
                "engine": engine,
                "mileage_km": mileage,
                "city": city,
                "url": link
            })

        print(f"Səhifə {page} tamamlandı | Cəmi toplanan: {len(cars_data)} / {target_count}")
        
        # Hər 10 səhifədən bir checkpoint kimi saxlayırıq
        if page % 10 == 0:
            pd.DataFrame(cars_data).to_csv(output_file, index=False, encoding="utf-8-sig")
            print(f"--- Checkpoint: {len(cars_data)} elan '{output_file}' faylına saxlanıldı ---")

        page += 1
        time.sleep(0.4)

    df = pd.DataFrame(cars_data)
    df.to_csv(output_file, index=False, encoding="utf-8-sig")
    print(f"\nUğurla {len(df)} elan '{output_file}' faylına yazıldı!")
    return df

if __name__ == "__main__":
    scrape_turbo_5k(target_count=5000)

Hədəf: 5000 elan. Çəkilməyə başlanılır...

Səhifə 1 tamamlandı | Cəmi toplanan: 36 / 5000
Səhifə 2 tamamlandı | Cəmi toplanan: 72 / 5000
Səhifə 3 tamamlandı | Cəmi toplanan: 108 / 5000
Səhifə 4 tamamlandı | Cəmi toplanan: 144 / 5000
Səhifə 5 tamamlandı | Cəmi toplanan: 180 / 5000
Səhifə 6 tamamlandı | Cəmi toplanan: 216 / 5000
Səhifə 7 tamamlandı | Cəmi toplanan: 252 / 5000
Səhifə 8 tamamlandı | Cəmi toplanan: 288 / 5000
Səhifə 9 tamamlandı | Cəmi toplanan: 324 / 5000
Səhifə 10 tamamlandı | Cəmi toplanan: 360 / 5000
--- Checkpoint: 360 elan 'turbo_az_5k_dataset.csv' faylına saxlanıldı ---
Səhifə 11 tamamlandı | Cəmi toplanan: 396 / 5000
Səhifə 12 tamamlandı | Cəmi toplanan: 432 / 5000
Səhifə 13 tamamlandı | Cəmi toplanan: 468 / 5000
Səhifə 14 tamamlandı | Cəmi toplanan: 504 / 5000
Səhifə 15 tamamlandı | Cəmi toplanan: 540 / 5000
Səhifə 16 tamamlandı | Cəmi toplanan: 576 / 5000
Səhifə 17 tamamlandı | Cəmi toplanan: 612 / 5000
Səhifə 18 tamamlandı | Cəmi toplanan: 648 / 5000
Səhifə 19 ta

In [67]:
import pandas as pd

df = pd.read_csv("turbo_az_5k_dataset.csv")
print(f"Cəmi toplanan təmiz elan sayı: {len(df)}")

Cəmi toplanan təmiz elan sayı: 5000


In [68]:
df

,make,model,price_value,price_currency,year,engine,mileage_km,city,url
0,Lynk,& Co 900,104900.0,AZN,2026,1.5 L,0,Bakı,https://turbo.az/autos/10010152-lynk-co-900
1,Li,Auto L9,125800.0,AZN,2025,1.5 L,0,Bakı,https://turbo.az/autos/10057115-li-auto-l9
2,CFMOTO,450 CL-C BOBBER,10300.0,AZN,2025,NaN,99,Bakı,https://turbo.az/autos/10565336-cfmoto-450-cl-...
3,Hyundai,Avante,21000.0,AZN,2015,1.6 L,137000,Bakı,https://turbo.az/autos/10633040-hyundai-avante
4,Land,Rover Defender,121000.0,AZN,2021,3.0 L,136000,Bakı,https://turbo.az/autos/10320565-land-rover-def...
...,...,...,...,...,...,...,...,...,...
4995,Mercedes,190,5800.0,AZN,1992,2.0 L,700000,Qusar,https://turbo.az/autos/10635469-mercedes-190
4996,Ford,Transit,18800.0,AZN,2009,2.2 L,173000,Bakı,https://turbo.az/autos/10595807-ford-transit
4997,Changan,Qiyuan Q05,32300.0,AZN,2025,1.5 L,0,Bakı,https://turbo.az/autos/10633086-changan-qiyuan...
4998,Changan,CS 75 Pro,32500.0,AZN,2026,1.5 L,0,Bakı,https://turbo.az/autos/10555865-changan-cs-75-pro


In [69]:
df = df.drop_duplicates()

In [70]:
df.shape

(4078, 9)

In [71]:
df.isnull().sum()

make                0
model               0
price_value         0
price_currency      0
year                0
engine            140
mileage_km          0
city                0
url                 0
dtype: int64

In [72]:
df.dropna(subset=['engine'], inplace=True)

In [73]:
df.isnull().sum()

make              0
model             0
price_value       0
price_currency    0
year              0
engine            0
mileage_km        0
city              0
url               0
dtype: int64

In [74]:
df.head()

,make,model,price_value,price_currency,year,engine,mileage_km,city,url
0,Lynk,& Co 900,104900.0,AZN,2026,1.5 L,0,Bakı,https://turbo.az/autos/10010152-lynk-co-900
1,Li,Auto L9,125800.0,AZN,2025,1.5 L,0,Bakı,https://turbo.az/autos/10057115-li-auto-l9
3,Hyundai,Avante,21000.0,AZN,2015,1.6 L,137000,Bakı,https://turbo.az/autos/10633040-hyundai-avante
4,Land,Rover Defender,121000.0,AZN,2021,3.0 L,136000,Bakı,https://turbo.az/autos/10320565-land-rover-def...
5,Audi,A8,8800.0,AZN,2003,3.7 L,173000,Bakı,https://turbo.az/autos/10464663-audi-a8


In [75]:
df['price_currency'].unique()

<ArrowStringArray>
['AZN']
Length: 1, dtype: str

In [76]:
df.drop(columns=["price_currency"], inplace=True)

In [77]:
df['car_age'] = 2026 - df['year']
df['car_age'] = df['car_age'].apply(lambda x: 1 if x <= 0 else x)

In [78]:
import numpy as np

In [79]:
df['is_new'] = np.where(df['mileage_km'] == 0, True, False)

In [84]:
df.rename(columns={
    "make": "marka",
    "price_value": "price"
}, inplace=True)

In [85]:
bins = [0, 15000, 35000, 70000, np.inf]
labels = ['Büdcə (<15k)', 'Orta (15k-35k)', 'Biznes (35k-70k)', 'Lüks (70k+)']
df['price_segment'] = pd.cut(df['price'], bins=bins, labels=labels)

In [86]:
df.head()

,marka,model,price,year,engine,mileage_km,city,url,car_age,is_new,price_segment
0,Lynk,& Co 900,104900.0,2026,1.5 L,0,Bakı,https://turbo.az/autos/10010152-lynk-co-900,1,True,Lüks (70k+)
1,Li,Auto L9,125800.0,2025,1.5 L,0,Bakı,https://turbo.az/autos/10057115-li-auto-l9,1,True,Lüks (70k+)
3,Hyundai,Avante,21000.0,2015,1.6 L,137000,Bakı,https://turbo.az/autos/10633040-hyundai-avante,11,False,Orta (15k-35k)
4,Land,Rover Defender,121000.0,2021,3.0 L,136000,Bakı,https://turbo.az/autos/10320565-land-rover-def...,5,False,Lüks (70k+)
5,Audi,A8,8800.0,2003,3.7 L,173000,Bakı,https://turbo.az/autos/10464663-audi-a8,23,False,Büdcə (<15k)


In [88]:
real_errors = df[(df['price'] < 1000) | (df['mileage_km'] > 1000000)]
print("Mütləq silinməli olan xətalı sətir sayı:", len(real_errors))

Mütləq silinməli olan xətalı sətir sayı: 4


In [90]:
df = df[~((df['price'] < 1000) | (df['mileage_km'] > 1000000))]

In [91]:
df.shape

(3934, 11)

In [92]:
df.head()

,marka,model,price,year,engine,mileage_km,city,url,car_age,is_new,price_segment
0,Lynk,& Co 900,104900.0,2026,1.5 L,0,Bakı,https://turbo.az/autos/10010152-lynk-co-900,1,True,Lüks (70k+)
1,Li,Auto L9,125800.0,2025,1.5 L,0,Bakı,https://turbo.az/autos/10057115-li-auto-l9,1,True,Lüks (70k+)
3,Hyundai,Avante,21000.0,2015,1.6 L,137000,Bakı,https://turbo.az/autos/10633040-hyundai-avante,11,False,Orta (15k-35k)
4,Land,Rover Defender,121000.0,2021,3.0 L,136000,Bakı,https://turbo.az/autos/10320565-land-rover-def...,5,False,Lüks (70k+)
5,Audi,A8,8800.0,2003,3.7 L,173000,Bakı,https://turbo.az/autos/10464663-audi-a8,23,False,Büdcə (<15k)


In [93]:
df['engine'] = df['engine'].str.replace('L', '', case=False).str.strip()

df['engine'] = pd.to_numeric(df['engine'], errors='coerce')

print(df['engine'].dtypes)
print(df['engine'].head())

float64
0    1.5
1    1.5
3    1.6
4    3.0
5    3.7
Name: engine, dtype: float64


In [94]:
df.dtypes

marka                 str
model                 str
price             float64
year                int64
engine            float64
mileage_km          int64
city                  str
url                   str
car_age             int64
is_new               bool
price_segment    category
dtype: object

In [95]:
pip install sqlalchemy psycopg2-binary

   ---------------------------------------- 0.0/2.8 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.8 MB ? eta -:--:--
   ---------------------- ----------------- 1.6/2.8 MB 8.3 MB/s eta 0:00:01
   ---------------------------------------- 2.8/2.8 MB 10.6 MB/s  0:00:00
Note: you may need to restart the kernel to use updated packages.


In [96]:
# price_segment sütununu mətne çeviririk
df['price_segment'] = df['price_segment'].astype(str)

In [97]:
from sqlalchemy import create_engine

# 'PAROLUNUZ' hissəsinə pgAdmin parolu yazın
# 'turbo_db' hissəsinə yaratdığınız bazanın adını yazın
user = "postgres"
password = "admin123"  # öz parolunuz
host = "localhost"
port = "5432"
db_name = "turbo_db"

# Bağlantı sətiri (Connection string)
db_url = f"postgresql://{user}:{password}@{host}:{port}/{db_name}"
engine = create_engine(db_url)

# DataFrame-i birbaşa cədvəl kimi daxil edirik
# if_exists='replace' -> cədvəl varsa yeniləyir, yoxdursa sıfırdan yaradır
df.to_sql(name='turbo_cars', con=engine, if_exists='replace', index=False)

print(" 3,934 sətirlik data PostgreSQL 'turbo_cars' cədvəlinə uğurla yükləndi!")

 3,934 sətirlik data PostgreSQL 'turbo_cars' cədvəlinə uğurla yükləndi!
